In [58]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sagorkumarmitra/nlp-shakespeare/shakespeare.txt


In [59]:
path = kagglehub.dataset_download("sagorkumarmitra/nlp-shakespeare")

In [60]:
import torch 
from torch import nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt

In [61]:
with open('/kaggle/input/datasets/sagorkumarmitra/nlp-shakespeare/shakespeare.txt','r',encoding='utf8') as f:
    text = f.read()

In [62]:
type(text)

str

In [63]:
print(text[:1000])


                     1
  From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee.


                     2
  When forty winters shall besiege thy brow,
  And dig deep trenches in thy beauty's field,
  Thy youth's proud livery so gazed on now,
  Will be a tattered weed of small worth held:  
  Then being asked, where all thy beauty lies,
  Where all the treasure of thy lusty days;
  To say within thine own deep su

In [64]:
len(text)

5445609

In [65]:
all_characters = set(text)

In [66]:
print(all_characters)

{'i', '9', '4', 'H', 'y', 't', 'E', 'N', 'W', 'Y', 'q', 'v', ' ', '}', '<', 'j', 'o', '(', 'a', ':', 'C', 's', '>', '3', '`', 'G', '6', '&', '_', 'S', 'u', 'c', 'r', 'd', '"', 'l', '1', 'U', 'Z', 'g', '5', '\n', ']', 'x', 'L', 'I', 'T', 'p', 'b', 'h', 'O', '0', 'V', ')', '8', '!', 'D', 'm', ';', "'", 'P', 'A', 'X', 'w', 'F', 'Q', '.', 'K', 'n', 'R', ',', 'B', 'k', 'z', 'e', '|', 'f', '2', 'J', '?', '7', 'M', '[', '-'}


In [67]:
len(all_characters)

84

In [68]:
# num - letter
decoder = dict(enumerate(all_characters))

In [69]:
encode = {char: ind for ind, char in decoder.items()}

In [70]:
print(decoder)

{0: 'i', 1: '9', 2: '4', 3: 'H', 4: 'y', 5: 't', 6: 'E', 7: 'N', 8: 'W', 9: 'Y', 10: 'q', 11: 'v', 12: ' ', 13: '}', 14: '<', 15: 'j', 16: 'o', 17: '(', 18: 'a', 19: ':', 20: 'C', 21: 's', 22: '>', 23: '3', 24: '`', 25: 'G', 26: '6', 27: '&', 28: '_', 29: 'S', 30: 'u', 31: 'c', 32: 'r', 33: 'd', 34: '"', 35: 'l', 36: '1', 37: 'U', 38: 'Z', 39: 'g', 40: '5', 41: '\n', 42: ']', 43: 'x', 44: 'L', 45: 'I', 46: 'T', 47: 'p', 48: 'b', 49: 'h', 50: 'O', 51: '0', 52: 'V', 53: ')', 54: '8', 55: '!', 56: 'D', 57: 'm', 58: ';', 59: "'", 60: 'P', 61: 'A', 62: 'X', 63: 'w', 64: 'F', 65: 'Q', 66: '.', 67: 'K', 68: 'n', 69: 'R', 70: ',', 71: 'B', 72: 'k', 73: 'z', 74: 'e', 75: '|', 76: 'f', 77: '2', 78: 'J', 79: '?', 80: '7', 81: 'M', 82: '[', 83: '-'}


In [71]:
print(encode)

{'i': 0, '9': 1, '4': 2, 'H': 3, 'y': 4, 't': 5, 'E': 6, 'N': 7, 'W': 8, 'Y': 9, 'q': 10, 'v': 11, ' ': 12, '}': 13, '<': 14, 'j': 15, 'o': 16, '(': 17, 'a': 18, ':': 19, 'C': 20, 's': 21, '>': 22, '3': 23, '`': 24, 'G': 25, '6': 26, '&': 27, '_': 28, 'S': 29, 'u': 30, 'c': 31, 'r': 32, 'd': 33, '"': 34, 'l': 35, '1': 36, 'U': 37, 'Z': 38, 'g': 39, '5': 40, '\n': 41, ']': 42, 'x': 43, 'L': 44, 'I': 45, 'T': 46, 'p': 47, 'b': 48, 'h': 49, 'O': 50, '0': 51, 'V': 52, ')': 53, '8': 54, '!': 55, 'D': 56, 'm': 57, ';': 58, "'": 59, 'P': 60, 'A': 61, 'X': 62, 'w': 63, 'F': 64, 'Q': 65, '.': 66, 'K': 67, 'n': 68, 'R': 69, ',': 70, 'B': 71, 'k': 72, 'z': 73, 'e': 74, '|': 75, 'f': 76, '2': 77, 'J': 78, '?': 79, '7': 80, 'M': 81, '[': 82, '-': 83}


In [72]:
encoded_text = np.array([encode[char] for char in text])

In [73]:
encoded_text[:900]

array([41, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,
       12, 12, 12, 12, 12, 36, 41, 12, 12, 64, 32, 16, 57, 12, 76, 18,  0,
       32, 74, 21,  5, 12, 31, 32, 74, 18,  5, 30, 32, 74, 21, 12, 63, 74,
       12, 33, 74, 21,  0, 32, 74, 12,  0, 68, 31, 32, 74, 18, 21, 74, 70,
       41, 12, 12, 46, 49, 18,  5, 12,  5, 49, 74, 32, 74, 48,  4, 12, 48,
       74, 18, 30,  5,  4, 59, 21, 12, 32, 16, 21, 74, 12, 57,  0, 39, 49,
        5, 12, 68, 74, 11, 74, 32, 12, 33,  0, 74, 70, 41, 12, 12, 71, 30,
        5, 12, 18, 21, 12,  5, 49, 74, 12, 32,  0, 47, 74, 32, 12, 21, 49,
       16, 30, 35, 33, 12, 48,  4, 12,  5,  0, 57, 74, 12, 33, 74, 31, 74,
       18, 21, 74, 70, 41, 12, 12,  3,  0, 21, 12,  5, 74, 68, 33, 74, 32,
       12, 49, 74,  0, 32, 12, 57,  0, 39, 49,  5, 12, 48, 74, 18, 32, 12,
       49,  0, 21, 12, 57, 74, 57, 16, 32,  4, 19, 41, 12, 12, 71, 30,  5,
       12,  5, 49, 16, 30, 12, 31, 16, 68,  5, 32, 18, 31,  5, 74, 33, 12,
        5, 16, 12,  5, 49

In [74]:
decoder[78]

'J'

In [75]:
def one_hot_encoder(encoded_text,num_uni_chars):
    # encoded_text > batch of encoded text
    # num_uni_chars > len(set(text))
    one_hot = np.zeros((encoded_text.size,num_uni_chars))#encoded text its a numpy array
    one_hot = one_hot.astype(np.float32)
    one_hot[np.arange(one_hot.shape[0]),encoded_text.flatten()] = 1.0
    one_hot =one_hot.reshape((*encoded_text.shape,num_uni_chars))
    return one_hot

In [76]:
def generate_batches(encoded_text,samp_per_batch=10,seq_len=50):
    # amount of char per batch
    char_per_batch = samp_per_batch * seq_len
    num_batches_avail= int(len(encoded_text)/char_per_batch)
    # amount of batches we can make, given the len of encoded_text
    encoded_text = encoded_text[:num_batches_avail*char_per_batch]
    encoded_text = encoded_text.reshape((samp_per_batch,-1))

    for n in range(0,encoded_text.shape[1],seq_len):
        x = encoded_text[:,n:n+seq_len]
        #zeros array to the same shape as x
        y = np.zeros_like(x)

        try:
            y[:,:-1] = x[:,1:]
            y[:,-1] = encoded_text[:,n+seq_len]
        except:
            y[:,:-1] = x [:,1:]
            y[:,-1] = encoded_text[:,0]
        yield x,y
        

In [77]:
sample_text = encoded_text[:20]
sample_text

array([41, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,
       12, 12, 12])

In [78]:
sample_text = encoded_text[:20]
sample_text

array([41, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,
       12, 12, 12])

In [79]:
batch_generator = generate_batches(sample_text,samp_per_batch=2,seq_len=5)

In [80]:
x,y = next(batch_generator)

In [81]:
x,y

(array([[41, 12, 12, 12, 12],
        [12, 12, 12, 12, 12]]),
 array([[12, 12, 12, 12, 12],
        [12, 12, 12, 12, 12]]))

In [82]:
# cerating the model 
class CharModel(nn.Module):
    def __init__(self,all_chars,num_hidden=256,num_layers=4,drop_prob=0.5,use_gpu=False):

        super().__init__()
        self.drop_prob = drop_prob
        self.num_layers = num_layers
        self.num_hidden = num_hidden
        self.use_gpu = use_gpu

        self.all_chars = all_chars
        self.decoder = dict(enumerate(all_chars))
        self.encoder = {char:ind for ind,char in decoder.items()}

        self.lstm=nn.LSTM(len(self.all_chars),num_hidden,num_layers,dropout=drop_prob,batch_first=True)
        self.dropout = nn.Dropout(drop_prob)

        self.fc_linear = nn.Linear(num_hidden,len(self.all_chars))

    def forward(self,x,hidden):
        lstm_output, hidden = self.lstm(x,hidden)

        drop_output = self.dropout(lstm_output)

        drop_out = drop_output.contiguous().view(-1,self.num_hidden)

        final_out = self.fc_linear(drop_output)
        return final_out,hidden

    def hidden_state (self,batch_size):
        if self.use_gpu:
            hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden).cuda(),
                      torch.zeros(self.num_layers,batch_size,self.num_hidden).cuda())
        else:
            hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden),
                     torch.zeros(self.num_layers,batch_size,self.num_hidden))
        return hidden

In [83]:
  model = CharModel(all_chars = all_characters,
                   num_hidden = 512,
                   num_layers = 3,
                   drop_prob= 0.5,
                   use_gpu = True)

In [84]:
total_param=[]
for p in model.parameters():
    total_param.append(int(p.numel()))
print(sum(total_param))

5470292


In [85]:
optimizer= torch.optim.Adam(model.parameters(),lr=0.001)
criterion = nn.CrossEntropyLoss()

In [86]:
train_percent = 0.1

In [87]:
train_ind = int(len(encoded_text)*train_percent)

In [88]:
train_data = encoded_text[:train_ind]
val_data = encoded_text[train_ind:]

In [89]:
len(train_data)

544560

In [90]:
train_percent = 0.9
train_ind = int(len(encoded_text)* train_percent)
train_data = encoded_text[:train_ind]
val_data = encoded_text[train_ind:]

In [91]:
epochs = 60
batch_size = 100

seq_len=100

tracker =0

num_char = max(encoded_text)+1

In [ ]:
#set model to train
model.train()

# check to see if using gpu

if model.use_gpu:
    model.cuda()
for i in range(epochs):

    hidden = model.hidden_state(batch_size)

    for x,y in generate_batches(train_data,batch_size,seq_len):

        tracker +=1

        #one hot encode incoming data
        x = one_hot_encoder(x,num_char)

        # numpy array to tensor 
        inputs = torch.from_numpy(x)
        targets = torch.from_numpy(y)

        # adjust for gpu if necesary
        if model.use_gpu:
            inputs = inputs.cuda()
            targets = targets.cuda()

        # reset hidden state 
        # if we dont reset we would backprogate throught all training history
        hidden = tuple([state.data for state in hidden])

        model.zero_grad()

        lstm_output, hidden = model.forward(inputs,hidden)
        lstm_output = lstm_output.contiguous().view(-1, num_char)
        targets = targets.view(-1).long()



        loss =criterion(lstm_output, targets)

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(),max_norm=5)

        optimizer.step()

        #check validation set

        if tracker % 25 ==0:
            val_hidden = model.hidden_state(batch_size)
            val_losses = []
            model.eval()

        for x,y in generate_batches(val_data,batch_size,seq_len):

            # one hot encode incoming data
            x= one_hot_encoder(x,num_char)

            # numpy to tensor
            inputs = torch.from_numpy(x)
            targets = torch.from_numpy(y)

            # adjust for gpu if necessary 
            if model.use_gpu:
                inputs = inputs.cuda()
                targets = targets.cuda()
            # resset hidden state
            val_hidden= tuple([state.data for state in val_hidden])

            lstm_output, val_hidden = model.forward(inputs, val_hidden)
            lstm_output = lstm_output.contiguous().view(-1, num_char)
            targets = targets.view(-1).long()

val_loss = criterion(lstm_output, targets)
            val_losses.append(val_loss.item())

        # reset to training after val for loop
        model.train()

        print(f"Epochs: {i} Step: {tracker} Val Loss:{val_loss.item()}")

ValueError: Expected input batch_size (100) to match target batch_size (10000).